# Data Preparation: Combining All Three Project Datasets
**Rupal Paliwal, Mod C, Semester 1**

My capstone project (*Movie Consumer Behavior & Film Profitability Analysis*) comes with three datasets. This notebook joins them into one table with one row per film, which the weekly notebooks then use.

| Dataset | Files used | What it adds |
|---|---|---|
| Movies Box Office 2000 to 2024 (Kaggle) | `enhanced_box_office_data_2000-2024_u.csv` | Revenue, genres, rating, vote count, language |
| MovieLens 20M (GroupLens) | `movie.csv`, `rating.csv`, `tag.csv`, `genome_scores.csv`, `genome_tags.csv` | How many MovieLens users rated and tagged each film, their average rating, and 1,128 "tag genome" scores describing each film's content |
| 4 Services Streaming (Kaggle) | `all_streaming.csv` | Whether the film is on Netflix, Prime Video, Hulu, or Disney+ |

**How films are matched:** the datasets share no ID, so I match on a cleaned title (lower case, no punctuation or accents, "Matrix, The" turned into "The Matrix") plus release year. If there is no exact match I allow the year to differ by one, since datasets sometimes record festival vs. wide release years.

**Put all the files in the same folder as this notebook before running it.** `rating.csv` has 20 million rows, so it is read in chunks (about 30 seconds).

In [1]:
import pandas as pd
import numpy as np
import re
import unicodedata

## 1. Title cleaning helpers

In [2]:
def clean_title(t):
    t = unicodedata.normalize('NFKD', str(t)).encode('ascii', 'ignore').decode()
    t = t.lower().replace('&', ' and ')
    t = re.sub(r'[^a-z0-9 ]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def split_movielens_title(raw):
    # 'Matrix, The (1999)' -> ('The Matrix', 1999)
    m = re.match(r'^(.*)\((\d{4})\)\s*$', raw.strip())
    if not m:
        return raw, np.nan
    title, year = m.group(1).strip(), int(m.group(2))
    title = re.sub(r'\s*\([^()]*\)\s*$', '', title).strip()          # drop alternate title, e.g. '(a.k.a. ...)'
    m2 = re.match(r"^(.*), (The|A|An|Les|Le|La|L'|Il|El|Das|Der|Die)$", title)
    if m2:
        title = f'{m2.group(2)} {m2.group(1)}'
    return title, year

def match_on_title_year(left, right, right_cols):
    # Exact year first, then year - 1, then year + 1. Each box office film gets at most one match.
    out = pd.DataFrame(index=left.index, columns=right_cols, dtype=float)
    remaining = left.copy()
    for shift in [0, -1, 1]:
        r = right.assign(Year=right['year'] - shift)
        hit = remaining.reset_index().merge(r, on=['key', 'Year'], how='inner').drop_duplicates('index').set_index('index')
        out.loc[hit.index, right_cols] = hit[right_cols].values
        remaining = remaining.drop(hit.index)
    return out

## 2. Box office (base table)

In [3]:
bo = pd.read_csv('enhanced_box_office_data_2000-2024_u.csv')
bo['key'] = bo['Release Group'].map(clean_title)
print(bo.shape)

(5000, 14)


## 3. MovieLens 20M

In [4]:
movies = pd.read_csv('movie.csv')
parts = movies['title'].map(split_movielens_title)
movies['key'] = parts.str[0].map(clean_title)
movies['year'] = parts.str[1]
movies = movies.dropna(subset=['year'])
movies['year'] = movies['year'].astype(int)

# Ratings: count and average per movie, read in chunks (20 million rows)
counts, sums = [], []
for chunk in pd.read_csv('rating.csv', usecols=['movieId', 'rating'], chunksize=4_000_000):
    g = chunk.groupby('movieId')['rating']
    counts.append(g.size()); sums.append(g.sum())
n_ratings = pd.concat(counts).groupby(level=0).sum()
mean_rating = pd.concat(sums).groupby(level=0).sum() / n_ratings

tags = pd.read_csv('tag.csv')
n_tags = tags.groupby('movieId').size()

movies['ml_n_ratings'] = movies['movieId'].map(n_ratings).fillna(0)
movies['ml_mean_rating'] = movies['movieId'].map(mean_rating)
movies['ml_n_tags'] = movies['movieId'].map(n_tags).fillna(0)
# If two MovieLens entries share a title and year, keep the one with more ratings
movies = movies.sort_values('ml_n_ratings', ascending=False).drop_duplicates(['key', 'year'])
print('MovieLens movies:', len(movies), '| ratings:', int(n_ratings.sum()))

MovieLens movies: 27221 | ratings: 20000263


In [5]:
ml_cols = ['movieId', 'ml_n_ratings', 'ml_mean_rating', 'ml_n_tags']
ml_match = match_on_title_year(bo, movies[['key', 'year'] + ml_cols], ml_cols)
ml_match = ml_match.rename(columns={'movieId': 'ml_movieId'})
print('Box office films matched to MovieLens:', ml_match['ml_movieId'].notna().sum(),
      'of', (bo['Year'] <= 2015).sum(), 'films released 2000 to 2015')

Box office films matched to MovieLens: 2429 of 3200 films released 2000 to 2015


## 4. Streaming catalogs (Netflix, Prime Video, Hulu, Disney+)

In [6]:
st = pd.read_csv('all_streaming.csv')
st = st[st['movie_or_serie'] == 'Movie'].copy()
service = {'netflix-shows': 'on_netflix', 'amazon-prime-movies-and-tv-shows': 'on_prime',
           'hulu-movies-and-tv-shows': 'on_hulu', 'disney-movies-and-tv-shows': 'on_disney'}
st['service'] = st['channel_streaming'].map(service)
st['key'] = st['title'].map(clean_title)
st['year'] = st['release_year']

# One row per film with a 0/1 column per service
avail = (pd.crosstab([st['key'], st['year']], st['service']).clip(upper=1).reset_index())
svc_cols = list(service.values())
st_match = match_on_title_year(bo, avail[['key', 'year'] + svc_cols], svc_cols).fillna(0).astype(int)
st_match['n_services'] = st_match[svc_cols].sum(axis=1)
print('Box office films found on at least one service:', (st_match['n_services'] > 0).sum(), 'of', len(bo))
st_match[svc_cols].sum()

Box office films found on at least one service: 1168 of 5000


on_netflix    644
on_prime      212
on_hulu       215
on_disney     183
dtype: int64

## 5. Combine and save

In [7]:
combined = pd.concat([bo.drop(columns='key'), ml_match, st_match], axis=1)
combined['in_movielens'] = combined['ml_movieId'].notna().astype(int)
combined.to_csv('combined_films.csv', index=False)
print(combined.shape)
combined[['Release Group', 'Year', 'ml_n_ratings', 'ml_mean_rating', 'n_services', 'on_netflix', 'on_disney']].head()

(5000, 23)


,Release Group,Year,ml_n_ratings,ml_mean_rating,n_services,on_netflix,on_disney
0,Mission: Impossible II,2000,16230.0,2.961368,0,0,0
1,Gladiator,2000,32878.0,3.952248,0,0,0
2,Cast Away,2000,18913.0,3.616454,0,0,0
3,What Women Want,2000,8554.0,3.136895,0,0,0
4,Dinosaur,2000,2310.0,3.099784,1,0,1


## 6. MovieLens tag genome for the matched films

The tag genome scores how strongly each of 1,128 tags (for example "twist ending", "visually appealing", "based on a book") describes a movie, from 0 to 1. This gives a high dimensional description of each film's content, which is useful for the regularization (Week 2) and dimension reduction (Week 3) methods.

In [8]:
genome = pd.read_csv('genome_scores.csv')
genome_tags = pd.read_csv('genome_tags.csv')
matched_ids = combined['ml_movieId'].dropna().astype(int).unique()
g = genome[genome['movieId'].isin(matched_ids)]
wide = g.pivot(index='movieId', columns='tagId', values='relevance')
wide.columns = ['tag_' + genome_tags.set_index('tagId').loc[c, 'tag'] for c in wide.columns]
wide.to_csv('genome_features.csv')
print('Matched films with genome scores:', wide.shape[0], '| genome tags:', wide.shape[1])

Matched films with genome scores: 2196 | genome tags: 1128


## 7. Column count across the datasets used

In [9]:
files = {'Box office': ['enhanced_box_office_data_2000-2024_u.csv'],
         'MovieLens': ['movie.csv', 'rating.csv', 'tag.csv', 'genome_scores.csv', 'genome_tags.csv'],
         'Streaming': ['all_streaming.csv']}
rows = []
for ds, fs in files.items():
    for f in fs:
        rows.append({'dataset': ds, 'file': f, 'columns': len(pd.read_csv(f, nrows=1).columns)})
cols = pd.DataFrame(rows)
print('Raw columns across files used:', cols['columns'].sum())
print('Feature columns available for modeling (combined table + genome tags):', combined.shape[1] + wide.shape[1])
cols

Raw columns across files used: 40
Feature columns available for modeling (combined table + genome tags): 1151


,dataset,file,columns
0,Box office,enhanced_box_office_data_2000-2024_u.csv,13
1,MovieLens,movie.csv,3
2,MovieLens,rating.csv,4
3,MovieLens,tag.csv,4
4,MovieLens,genome_scores.csv,3
5,MovieLens,genome_tags.csv,2
6,Streaming,all_streaming.csv,11
